# Data Ingestion

This notebook demonstrates the **ingestion layer** of Equity Lake:

- Structured logging with `@timed` decorator and `timer()` context manager
- Sequential vs parallel market fetching performance comparison
- Gap detection and auto-backfill workflows
- Coverage statistics per market

**Prerequisites:** Ensure `config/settings.yaml` is configured with API keys for your target markets.

In [1]:
import sys
sys.path.insert(0, "../src")

from equity_lake.core.logging import (
    setup_structured_logging,
    timed,
    timer,
    correlation_context,
    get_correlation_id,
)
from equity_lake.ingestion.parallel import fetch_markets_parallel
from equity_lake.ingestion.gap_detection import GapDetector
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import time
from datetime import date, timedelta

print("Ingestion modules loaded successfully")

Ingestion modules loaded successfully


## 1. Structured Logging

Equity Lake uses `structlog` with JSON output, correlation IDs, and automatic timing decorators.

In [2]:
logger = setup_structured_logging(level="INFO", json_output=True, console=True)

# @timed decorator — automatically logs execution time
@timed(operation="demo_fetch", market="us")
def fetch_with_timing():
    time.sleep(0.5)
    return {"data": "sample", "rows": 100}

result = fetch_with_timing()
print(f"\nResult: {result}")

{"function": "fetch_with_timing", "duration_seconds": 0.505, "status": "success", "operation": "demo_fetch", "market": "us", "event": "fetch_with_timing_completed", "logger": "equity_lake.core.logging", "level": "info", "correlation_id": "4697af1a", "timestamp": "2026-06-09T01:31:07.961277Z"}



Result: {'data': 'sample', 'rows': 100}


In [3]:
# timer context manager — for batch operations
with timer("batch_processing", record_count=5000):
    time.sleep(0.3)
    print("  Batch processing completed")

# correlation context — track a full pipeline run
with correlation_context("ingestion-demo-001"):
    logger.info("ingestion_started", markets=["us", "cn"])
    logger.info("ingestion_step", step="fetch")
    print(f"  Correlation ID: {get_correlation_id()}")

  Batch processing completed
{"operation": "batch_processing", "duration_seconds": 0.303, "record_count": 5000, "event": "batch_processing_completed", "logger": "equity_lake.core.logging", "level": "info", "correlation_id": "4697af1a", "timestamp": "2026-06-09T01:31:08.271172Z"}


{"markets": ["us", "cn"], "event": "ingestion_started", "logger": "__main__", "level": "info", "correlation_id": "ingestion-demo-001", "timestamp": "2026-06-09T01:31:08.272029Z"}


{"step": "fetch", "event": "ingestion_step", "logger": "__main__", "level": "info", "correlation_id": "ingestion-demo-001", "timestamp": "2026-06-09T01:31:08.272394Z"}


  Correlation ID: ingestion-demo-001


## 2. Sequential vs Parallel Fetching

Compare performance of sequential vs parallel market fetching. We use mock functions to simulate API latency.

In [4]:
def mock_fetch_us(trading_date):
    time.sleep(1.5)
    return None

def mock_fetch_cn(trading_date):
    time.sleep(1.5)
    return None

def mock_fetch_hk_sg(trading_date):
    time.sleep(1.5)
    return None

fetch_func_map = {
    "us": (mock_fetch_us, {}),
    "cn": (mock_fetch_cn, {}),
    "hk_sg": (mock_fetch_hk_sg, {}),
}
markets = ["us", "cn", "hk_sg"]
trading_date = date.today()

In [5]:
# Fetch markets with parallel execution
print("Running parallel fetch...")
start = time.time()
par_results = fetch_markets_parallel(
    markets=markets,
    trading_date=trading_date,
    fetch_func_map=fetch_func_map,
    max_workers=len(markets),
)
par_time = time.time() - start
print(f"Parallel: {par_time:.2f}s — {len(par_results)} results")
for r in par_results:
    if isinstance(r, str):
        print(f"  {r}")
    elif hasattr(r, 'success'):
        status = "OK" if r.success else f"FAIL: {r.error}"
        print(f"  {r.market}: {status}")
    else:
        print(f"  {r}")


Running parallel fetch...


Parallel: 1.51s — 3 results
  us
  cn
  hk_sg


In [6]:
# Simulate sequential timing (fetch one-by-one)
print("Running sequential fetch (one market at a time)...")
seq_start = time.time()
seq_results = []
for mkt in markets:
    t0 = time.time()
    result = fetch_markets_parallel(
        markets=[mkt],
        trading_date=trading_date,
        fetch_func_map=fetch_func_map,
        max_workers=1,
    )
    seq_results.extend(result)
seq_time = time.time() - seq_start
print(f"Sequential: {seq_time:.2f}s — {len(seq_results)} markets fetched")


Running sequential fetch (one market at a time)...


Sequential: 4.51s — 3 markets fetched


In [7]:
# Visualize the comparison
fig = go.Figure()
fig.add_trace(go.Bar(
    x=["Sequential (1 worker)", "Parallel (N workers)"],
    y=[seq_time, par_time],
    marker_color=["#ef4444", "#22c55e"],
    text=[f"{seq_time:.2f}s", f"{par_time:.2f}s"],
    textposition="outside",
))
speedup = seq_time / par_time if par_time > 0 else 0
fig.update_layout(
    title=f"Sequential vs Parallel Fetching — {speedup:.1f}x Speedup",
    yaxis_title="Time (seconds)",
    height=400,
    showlegend=False,
)
fig.show()


## 3. Gap Detection

The `GapDetector` uses DuckDB to find missing trading dates in the data lake.

In [8]:
try:
    with GapDetector() as detector:
        # Get coverage stats for each market
        stats_data = []
        for market in ["us_equity", "cn_ashare", "hk_sg_equity"]:
            try:
                coverage = detector.get_coverage_stats(market)
                stats_data.append({
                    "Market": market,
                    "Total Dates": coverage.get("total_dates", 0),
                    "Expected Dates": coverage.get("expected_dates", 0),
                    "Coverage %": coverage.get("coverage_pct", 0),
                    "Latest Date": str(coverage.get("latest_date", "N/A")),
                })
            except Exception as e:
                stats_data.append({"Market": market, "Error": str(e)})

        if stats_data:
            stats_df = pd.DataFrame(stats_data)
            print(stats_df.to_string(index=False))
        else:
            print("No coverage data available — run ingestion first: uv run equity ingest")
except Exception as e:
    print(f"GapDetector requires data in data/lake/: {e}")
    print("Run: dotenvx run -- uv run equity ingest")

      Market  Total Dates  Expected Dates  Coverage % Latest Date
   us_equity            0               0           0         N/A
   cn_ashare            0               0           0         N/A
hk_sg_equity            0               0           0         N/A


In [9]:
# Visualize coverage as a gauge chart
try:
    if stats_data and "Coverage %" in stats_df.columns:
        fig = go.Figure()
        for _, row in stats_df.iterrows():
            if "Error" not in row:
                fig.add_trace(go.Indicator(
                    mode="gauge+number",
                    value=row.get("Coverage %", 0),
                    title={"text": row["Market"]},
                    domain={"row": 0, "column": list(stats_df["Market"]).index(row["Market"])},
                    gauge={"axis": {"range": [0, 100]}},
                ))
        fig.update_layout(
            grid={"rows": 1, "columns": len(stats_df)},
            height=300,
            title="Market Data Coverage (%)",
        )
        fig.show()
except Exception as e:
    print(f"Coverage visualization: {e}")

## 4. Auto-Backfill

The ingestion layer can automatically detect and fill gaps in historical data.

In [10]:
from equity_lake.ingestion.auto_backfill import find_and_fill_gaps

# Find gaps without filling (dry run)
print("To auto-backfill missing dates:")
print("  dotenvx run -- uv run equity auto-backfill")
print()
print("To manually backfill a date range:")
print("  dotenvx run -- uv run equity backfill --start 2024-01-01 --end 2024-12-31")
print()
print("To detect gaps only:")
print("  dotenvx run -- uv run equity ingest --detect-gaps --coverage-stats")

To auto-backfill missing dates:
  dotenvx run -- uv run equity auto-backfill

To manually backfill a date range:
  dotenvx run -- uv run equity backfill --start 2024-01-01 --end 2024-12-31

To detect gaps only:
  dotenvx run -- uv run equity ingest --detect-gaps --coverage-stats


## 5. Market Data Freshness

Check when each market was last updated.

In [11]:
try:
    with GapDetector() as detector:
        freshness_data = []
        for market_dir in ["us_equity", "cn_ashare", "hk_sg_equity", "jpx_equity", "krx_equity"]:
            try:
                latest = detector.get_latest_date(market_dir)
                if latest:
                    days_old = (date.today() - latest).days
                    freshness_data.append({
                        "Market": market_dir,
                        "Latest Date": str(latest),
                        "Days Old": days_old,
                        "Status": "Fresh" if days_old <= 1 else ("Stale" if days_old <= 5 else "Critical"),
                    })
            except Exception:
                pass

        if freshness_data:
            fresh_df = pd.DataFrame(freshness_data)
            print(fresh_df.to_string(index=False))

            fig = px.bar(
                fresh_df, x="Market", y="Days Old", color="Status",
                color_discrete_map={"Fresh": "#22c55e", "Stale": "#f59e0b", "Critical": "#ef4444"},
                title="Data Freshness by Market",
            )
            fig.update_layout(height=400)
            fig.show()
        else:
            print("No market data found — run ingestion first")
except Exception as e:
    print(f"Freshness check: {e}")

No market data found — run ingestion first


## Next Steps

- **[03-storage-and-queries.ipynb](03-storage-and-queries.ipynb)** — Query ingested data with DuckDB
- **[04-hamilton-features.ipynb](04-hamilton-features.ipynb)** — Generate features from ingested data
- **[08-backtesting.ipynb](08-backtesting.ipynb)** — Backtest strategies on ingested data